In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def absmax_quantize(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    将浮点张量 X 量化为 INT8，并返回缩放因子。
    
    Args:
        x: 浮点类型的张量
    Returns:
        x_quant: dtype 为 torch.int8 的量化张量
        scale: float 类型的缩放因子
    """
    # ==========================================
    # TODO 1: 计算张量的绝对最大值 absmax
    # 提示: 使用 torch.abs 和 torch.max
    # ==========================================
    absmax = torch.max(torch.abs(x))
    
    # 避免除以 0 的情况
    if absmax == 0:
        absmax = 1e-8
        
    # ==========================================
    # TODO 2: 计算缩放因子 scale (映射到 [-127, 127])
    # ==========================================
    scale = 127/absmax
    
    # ==========================================
    # TODO 3: 量化过程
    # 1. 乘以 scale
    # 2. 四舍五入 (torch.round)
    # 3. 截断到 [-128, 127] (torch.clamp)
    # 4. 转换数据类型为 torch.int8 (x.to(torch.int8))
    # ==========================================
    x_scaled = x * scale
    x_quant = torch.clamp(torch.round(x_scaled),max=127,min=-128).to(torch.int8)
    
    return x_quant, scale
    pass

class W8A16Linear(nn.Module):
    """
    Weight-only INT8 量化线性层。
    在内存中，我们存储的是非常微小的 INT8 权重。
    在计算时，我们将权重反量化回 FP16，与同样是 FP16 的输入进行矩阵乘法。
    这种方式虽然没有加速计算，但极大地缓解了从内存读取权重的 Memory-bound (带宽高了 2 倍)。
    """
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        # 内存中存储的是 int8 格式
        self.register_buffer("weight_int8", torch.zeros((out_features, in_features), dtype=torch.int8))
        self.register_buffer("scale", torch.tensor(1.0))
        self.bias = nn.Parameter(torch.zeros(out_features))

    def from_float(self, linear_layer: nn.Linear):
        """
        从高精度的 Linear 层中吸收权重并进行 PTQ 量化
        """
        w_quant, scale = absmax_quantize(linear_layer.weight.data)
        self.weight_int8.copy_(w_quant)
        self.scale.copy_(scale)
        if linear_layer.bias is not None:
            self.bias.data.copy_(linear_layer.bias.data)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 4: 反量化与前向传播
        # 1. 将 weight_int8 转换回与输入 x 相同的类型 (如 float32/float16)
        # 2. 除以 self.scale 恢复其数值范围
        # 3. 使用 F.linear 进行标准的矩阵乘法
        # ==========================================
        # w_fp = self.weight_int8.type(x.type())
        w_fp = self.weight_int8.type_as(x)
        w_dequant = w_fp/self.scale
        
        out = F.linear(x,w_dequant)
        return out
        pass

/home/helen/miniforge3/envs/agent_langchain/lib/python3.11/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
# 测试你的实现
def test_quantization():
    try:
        torch.manual_seed(42)
        
        # 1. 测试 absmax_quantize
        x_fp = torch.tensor([-0.8, 1.5, -3.0, 2.5, 0.0])
        # 绝对最大值是 3.0。Scale = 127 / 3.0 = 42.333
        # 2.5 * 42.333 = 105.8 -> 106
        x_q, scale = absmax_quantize(x_fp)
        
        assert x_q.dtype == torch.int8, "量化后的张量必须是 int8 类型！"
        assert torch.allclose(scale, torch.tensor(127.0 / 3.0)), "Scale 计算不正确！"
        assert x_q[3].item() == 106, "量化后的四舍五入数值计算不正确！"
        print("✅ absmax_quantize 核心算法测试通过！")
        
        # 2. 测试 W8A16 线性层
        in_dim, out_dim = 128, 64
        batch, seq = 2, 10
        
        # 构建一个原始的 FP32 Linear 层
        fp_linear = nn.Linear(in_dim, out_dim)
        
        # 构建我们的 INT8 量化层并吸入权重
        q_linear = W8A16Linear(in_dim, out_dim)
        q_linear.from_float(fp_linear)
        
        # 验证显存占用 (理论上应该小 4 倍，因为 FP32 是 4 字节，INT8 是 1 字节)
        fp_bytes = fp_linear.weight.element_size() * fp_linear.weight.numel()
        q_bytes = q_linear.weight_int8.element_size() * q_linear.weight_int8.numel()
        
        assert q_bytes == fp_bytes // 4, "INT8 权重的内存占用必须是 FP32 的四分之一！"
        
        # 验证前向传播结果的误差 (量化必然带来微小误差)
        x_input = torch.randn(batch, seq, in_dim)
        out_fp = fp_linear(x_input)
        out_q = q_linear(x_input)
        # 计算余弦相似度，因为经过量化反量化，数值不可能完全一致，只要相似度极高就算成功
        cos_sim = F.cosine_similarity(out_fp.flatten(), out_q.flatten(), dim=0)
        assert cos_sim > 0.99, f"反量化计算出的张量与原始张量差异过大，相似度仅为: {cos_sim.item():.4f}"
        
        print(f"✅ W8A16Linear 测试通过！输出相似度极高 (Cosine Sim: {cos_sim.item():.4f})，且权重内存成功缩小 4 倍！")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
    except AttributeError:
        print("代码未完成导致变量属性错误。")
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
    except Exception as e:
        print(f"❌ 发生未知异常: {e}")

test_quantization()

✅ absmax_quantize 核心算法测试通过！
✅ W8A16Linear 测试通过！输出相似度极高 (Cosine Sim: 0.9957)，且权重内存成功缩小 4 倍！
